# 03 - Grid Events Silver
### Goal
- import reusable code from src/
- use df.transform()
- apply UDF for duration classification
- write silver Delta table

In [0]:
%run ../01_setup/00_config

In [0]:
from transforms.grid_events_cleaning import standardize_grid_events_columns, filter_invalid_grid_events_rows
from transforms.business_rules import add_grid_events_flags
from udfs.grid_events_udfs import classify_duration_band

In [0]:
bronze_df = spark.table(bronze_grid_events_table)

silver_df = (
    bronze_df
    .transform(standardize_grid_events_columns)
    .transform(lambda df: filter_invalid_grid_events_rows(df, rules))
    .transform(lambda df: add_grid_events_flags(df, rules))
    .withColumn("duration_band", classify_duration_band(F.col("duration_minutes")))
    .drop("_rescued_data")
)

display(silver_df.limit(10))
print(f"Bronze rows: {bronze_df.count()} - Silver rows: {silver_df.count()}")

In [0]:
silver_df.write.mode("overwrite").saveAsTable(silver_grid_events_table)
print(f"Silver table saved: {silver_grid_events_table}")